# Exploratory Data Analysis (EDA) & Problem Insights
This notebook explores the semiconductor dataset, analyzes the physical characteristics of the multiplicative speckle noise, and establishes the baseline image quality metrics before any deep learning restoration is applied.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Load file paths
noisy_files = sorted(glob.glob("../train/NoisyLR/*.npy"))
gt_files = sorted(glob.glob("../train/GT/*.npy"))
print(f"Dataset Size: {len(noisy_files)} Image Pairs")


## 1. Visualizing the Degradation
Let's pull a random image pair from the dataset to visually inspect the difference between the Ground Truth (Clean) and the Noisy Low-Resolution input.

In [ ]:
idx = 42
noisy_img = np.load(noisy_files[idx]).astype(np.float32)
gt_img = np.load(gt_files[idx]).astype(np.float32)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(noisy_img, cmap='gray')
plt.title("Noisy LR Input")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(gt_img, cmap='gray')
plt.title("Ground Truth (Clean HR)")
plt.axis('off')

plt.tight_layout()
plt.show()


## 2. Noise Distribution Analysis
Speckle noise in imaging is often multiplicative rather than additive (e.g., Gaussian). Let's calculate the physical noise mask by subtracting the Ground Truth from the Noisy image and looking at its distribution.

In [ ]:
noise_mask = noisy_img - gt_img

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.imshow(noise_mask, cmap='coolwarm')
plt.title("Extracted Noise Mask (Noisy - GT)")
plt.colorbar()
plt.axis('off')

plt.subplot(1, 2, 2)
plt.hist(noise_mask.flatten(), bins=100, color='purple', alpha=0.7)
plt.title("Noise Pixel Value Distribution")
plt.xlabel("Pixel Intensity Difference")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Baseline Metrics (Noisy vs GT)
Before we can evaluate how good our `EDSR50` model is, we need to know how bad the images are to begin with. We will calculate the average PSNR and SSIM of the raw, untouched Noisy images against the Ground Truth across 500 samples.

In [ ]:
num_samples = min(500, len(noisy_files))
base_ssim = []
base_psnr = []

for i in range(num_samples):
    n_arr = np.load(noisy_files[i]).astype(np.float32)
    g_arr = np.load(gt_files[i]).astype(np.float32)
    
    base_ssim.append(ssim(g_arr, n_arr, data_range=1.0))
    base_psnr.append(psnr(g_arr, n_arr, data_range=1.0))

print("="*40)
print("BASELINE METRICS (UNTRAINED)")
print("="*40)
print(f"Average Baseline SSIM : {np.mean(base_ssim):.4f}")
print(f"Average Baseline PSNR : {np.mean(base_psnr):.2f} dB")
print("="*40)
print("Our EDSR50 model must significantly beat these baseline scores to prove it is learning to invert the speckle noise.")


## 4. Our Approach: The EDSR50 Architecture
To solve this, we implemented a custom variant of the **Enhanced Deep Residual Networks (EDSR)** architecture.

### Why EDSR?
Traditional networks like U-Net rely heavily on spatial downsampling (pooling/strided convolutions) to capture global context. However, speckle noise lives at the extremely high-frequency sub-pixel level. When a U-Net downsamples the image, it physically destroys this high-frequency noise data (the "Compression Trap"). 

EDSR avoids this trap entirely by maintaining the full spatial resolution throughout the entire network using pure Residual Blocks.

### Novelty: Decoupled Multi-Task Heads
Instead of forcing a single layer to figure out how to both Denoise AND Upscale simultaneously, our `MultiTaskEDSR` architecture physically splits the final layers into decoupled branches:
1. **Denoising Branch:** Focuses purely on mathematically smoothing the multiplicative speckle.
2. **Super-Resolution Branch:** Focuses purely on interpolating sub-pixel edges using PixelShuffle.
3. **Fusion Layer:** Recombines the clean semantic structure with the sharp upscaled edges to produce the final `512x512` output.